In [20]:
import os
import glob
import h5py
import numpy as np
from astropy.table import Table, vstack

def h5_fits_crossmatch(fits_path, script_dir=None, cluster=None):

    if script_dir is None:
        script_dir = os.getcwd()

    # Collects all the files ending in .h5 and returns them as a list of file paths
    h5_files = glob.glob(os.path.join(script_dir, "Reduced Spectra", "*.h5"))

    # Keeps only files that contain the number stored in cluster
    if cluster is not None:
        h5_files = [f for f in h5_files if str(cluster) in os.path.basename(f)]

    if not h5_files:
        raise ValueError("No H5 files found.")

    # Creates a table for the h5 files and loops over every h5 file
    h5_tables = []
    for fpath in h5_files:
        with h5py.File(fpath, "r") as f:
            h5_tables.append(Table(f["catalog"][:]))

    # Stacks the tables vertically into one big catalog
    h5 = vstack(h5_tables)

    # unique H5 stars
    _, idx = np.unique(h5["GAIAEDR3_ID"], return_index=True)
    h5_ids = np.array(h5["GAIAEDR3_ID"][idx]).astype(str)

    # Load FITS catalog
    fits = Table.read(fits_path)

    # correct Gaia ID column in your file
    fits_ids = np.array(fits["GAIADER3_ID"]).astype(str)

    # For each h5 star, check if the ID exists in fits. Produces a boolean array that is true if found in fits, false if it is missing
    fits_set = set(fits_ids)
    in_fits = np.array([i in fits_set for i in h5_ids])
    not_in_fits = ~in_fits

    # Print out the results
    total = len(h5_ids)
    matched = np.sum(in_fits)
    missing = np.sum(not_in_fits)

    print("-- H5 vs FITS crossmatch --")
    print(f"Total unique H5 stars: {total}")
    print(f"In FITS:       {matched}")
    print(f"Missing in FITS:       {missing}")

    print("\nMissing IDs:")
    print(h5_ids[not_in_fits][:20])


In [21]:
h5_fits_crossmatch("NGC_6811.fits", cluster="6811")

-- H5 vs FITS crossmatch --
Total unique H5 stars: 2716
In FITS:       2716
Missing in FITS:       0

Missing IDs:
[]
